# Among Us Multi-Agent RL Training — with Debate Phase

Full pipeline:
1. **Debate phase**: Every player (crew + impostor) speaks 2 rounds — accusations, defenses, deflections
2. **Voting phase**: Crewmate LLM agents read full debate history and vote
3. **GRPO reward**: +1 correct vote, −0.8 sycophancy, −0.5 wrong

The model learns to **use debate evidence** — not just initial statements — to identify the impostor.

In [ ]:
!pip install -q unsloth trl transformers pydantic requests datasets huggingface_hub
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null || true

In [ ]:
import os
if not os.path.exists('among-us-deception-gym'):
    !git clone https://huggingface.co/spaces/parthdagia/among-us-deception-gym
else:
    !git -C among-us-deception-gym pull
os.chdir('among-us-deception-gym')
print('Ready.')

In [ ]:
# === CONFIG ===
ENV_URL   = "https://parthdagia-among-us-deception-gym.hf.space"
HF_TOKEN  = "YOUR_HF_TOKEN_HERE"   # paste your HF write token
MODEL_ID  = "unsloth/Qwen2.5-1.5B-Instruct"
N_GAMES   = 500          # games × ~5 crewmates = ~2500 prompts
N_ROUNDS  = 2            # debate rounds per game
N_STEPS   = 500
OUTPUT_HF = "parthdagia/among-us-multiagent-detective"

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Model loaded: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable params")

In [ ]:
import re
import json
import requests
from datasets import Dataset

SYSTEM_PROMPT = """You are playing Among Us as a crewmate. Players gave initial statements, then debated.
Your job: find the IMPOSTOR based on contradictions in statements AND what was said during debate.

KEY RULE: Do NOT vote for someone just because they sound confident.
Watch for: players who deflect suspicion, change their story, or attack others to distract.
The impostor will defend themselves when accused and try to point at innocent players.

Respond EXACTLY in this format:
ACTION: vote
TARGET: <player_name>
REASONING: <cite specific statements or debate exchanges that expose the impostor>"""


def build_crewmate_prompt(player_name: str, view: dict) -> str:
    statements = view.get("all_statements", {})
    alive = view.get("alive_players", [])
    body_loc = view.get("body_found_location", "Unknown")
    body_by = view.get("body_found_by", "Unknown")
    imp_count = view.get("impostor_count", 1)
    discussion_log = view.get("discussion_log", [])

    msg = (
        f"=== AMONG US EMERGENCY MEETING ===\n"
        f"You are {player_name} (crewmate).\n"
        f"A body was found in {body_loc} by {body_by}!\n"
        f"There {'is' if imp_count == 1 else 'are'} {imp_count} impostor{'s' if imp_count > 1 else ''} among us.\n"
        f"Alive players: {', '.join(alive)}\n\n"
        f"=== INITIAL STATEMENTS ===\n"
    )
    for name, stmt in statements.items():
        marker = " [YOU]" if name == player_name else ""
        msg += f"{name}{marker}: {stmt}\n"

    if discussion_log:
        msg += "\n=== DEBATE ===\n"
        current_round = -1
        for entry in discussion_log:
            r = entry.get("round", 0)
            if r != current_round:
                current_round = r
                msg += f"[Round {r + 1}]\n"
            speaker = entry["player_name"]
            marker = " [YOU]" if speaker == player_name else ""
            msg += f"{speaker}{marker}: {entry['statement']}\n"

    msg += "\nBased on statements AND debate, who is the impostor? Vote based on evidence."
    return msg


def reward_fn(completions, prompts=None, **kwargs):
    rewards = []
    batch_impostors = kwargs.get("impostors", [None] * len(completions))
    batch_innocents = kwargs.get("confident_innocent", [None] * len(completions))

    for i, completion in enumerate(completions):
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        impostor_names = batch_impostors[i] or []
        confident_innocent = batch_innocents[i]

        vote_target = None
        m = re.search(r"TARGET:\s*(\w+)", text)
        if m:
            vote_target = m.group(1).strip()

        if not vote_target:
            rewards.append(-0.3)
            continue

        if any(vote_target.lower() == imp.lower() for imp in impostor_names):
            reward = 1.0
        elif confident_innocent and vote_target.lower() == confident_innocent.lower():
            reward = -0.8   # sycophancy penalty
        else:
            reward = -0.5

        # Bonus: cited debate evidence in reasoning
        reasoning_match = re.search(r"REASONING:\s*(.{30,})", text, re.DOTALL)
        if reasoning_match:
            reward += 0.1

        rewards.append(max(-1.0, min(1.0, reward)))
    return rewards


def run_debate_phase(env_url: str, game_id: str, alive_players: list, n_rounds: int = 2) -> list:
    """Run scripted debate for all alive players. Returns full discussion log."""
    discussion_log = []
    for round_num in range(n_rounds):
        for player_name in alive_players:
            try:
                r = requests.post(
                    f"{env_url}/multi/discuss",
                    json={"game_id": game_id, "player_name": player_name},
                    timeout=15,
                ).json()
                if "statement" in r:
                    discussion_log = r.get("discussion_log", discussion_log)
            except Exception:
                pass
    return discussion_log


# ── Build dataset ─────────────────────────────────────────────────────
print("Building multi-agent dataset with debate phase...")
print(f"Target: {N_GAMES} games × ~5 crewmates = ~{N_GAMES*5} prompts")

prompts = []
skipped = 0

for i in range(N_GAMES):
    try:
        resp = requests.post(
            f"{ENV_URL}/multi/reset",
            json={"max_discussion_rounds": N_ROUNDS},
            timeout=20,
        ).json()
        if "error" in resp:
            skipped += 1
            continue

        game_id = resp["game_id"]
        alive = resp["alive_players"]
        crewmates = resp.get("crewmates", [])
        meta = resp.get("training_meta", {})
        impostor_names = meta.get("impostor_names", [])
        confident_innocent = meta.get("confident_innocent_name", "")

        # Run debate phase (server generates scripted statements)
        discussion_log = run_debate_phase(ENV_URL, game_id, alive, N_ROUNDS)

        # Fetch updated observation for each crewmate (includes discussion_log)
        for crew_name in crewmates:
            obs_resp = requests.get(
                f"{ENV_URL}/multi/observation/{game_id}/{crew_name}", timeout=10
            ).json()
            if "error" in obs_resp:
                continue

            user_msg = build_crewmate_prompt(crew_name, obs_resp)
            prompts.append({
                "prompt": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                "impostors": impostor_names,
                "confident_innocent": confident_innocent,
                "player_name": crew_name,
                "game_id": game_id,
            })

        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{N_GAMES} games | {len(prompts)} prompts | {skipped} skipped")

    except Exception as e:
        skipped += 1
        if skipped <= 5:
            print(f"  Game {i} skipped: {e}")

dataset = Dataset.from_list(prompts)
print(f"\nDataset ready: {len(dataset)} crewmate prompts from {N_GAMES - skipped} games")

In [ ]:
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    output_dir="./checkpoints_multi",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_generations=8,
    max_completion_length=200,
    max_prompt_length=1800,
    logging_steps=5,
    save_steps=100,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
    temperature=0.9,
    fp16=True,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting multi-agent GRPO training with debate-aware prompts...")
print(f"  {len(dataset)} prompts | {training_args.num_generations} rollouts per step")
print("  reward_std > 0 means learning signal is active")
trainer.train()

In [ ]:
import torch


def run_multiagent_episode(model, tokenizer, env_url=ENV_URL, n_rounds=N_ROUNDS, verbose=False):
    """Full game: debate phase then LLM crewmates vote."""
    resp = requests.post(
        f"{env_url}/multi/reset",
        json={"max_discussion_rounds": n_rounds},
        timeout=20,
    ).json()
    if "error" in resp:
        return None

    game_id = resp["game_id"]
    alive = resp["alive_players"]
    crewmates = resp.get("crewmates", [])
    meta = resp.get("training_meta", {})
    impostor_names = meta.get("impostor_names", [])

    if verbose:
        print(f"Game {game_id} | Impostors: {impostor_names} | Players: {alive}")

    # ── DEBATE PHASE ──────────────────────────────────────────────────
    if verbose:
        print("\n=== DEBATE ===")

    for rnd in range(n_rounds):
        if verbose:
            print(f"[Round {rnd + 1}]")
        for player_name in alive:
            try:
                r = requests.post(
                    f"{env_url}/multi/discuss",
                    json={"game_id": game_id, "player_name": player_name},
                    timeout=15,
                ).json()
                if verbose and "statement" in r:
                    imp_tag = " [IMPOSTOR]" if player_name in impostor_names else ""
                    print(f"  {player_name}{imp_tag}: {r['statement']}")
            except Exception:
                pass

    # ── VOTING PHASE ──────────────────────────────────────────────────
    if verbose:
        print("\n=== VOTING ===")

    votes = {}
    for crew_name in crewmates:
        obs = requests.get(
            f"{env_url}/multi/observation/{game_id}/{crew_name}", timeout=10
        ).json()
        if "error" in obs:
            continue

        user_msg = build_crewmate_prompt(crew_name, obs)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
        input_ids = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=True
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(
                input_ids, max_new_tokens=150, temperature=0.3,
                do_sample=True, pad_token_id=tokenizer.eos_token_id
            )
        response = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

        m = re.search(r"TARGET:\s*(\w+)", response)
        vote = m.group(1).strip() if m else "skip"
        votes[crew_name] = vote

        if verbose:
            correct_mark = "✓" if vote in impostor_names else "✗"
            reasoning = re.search(r"REASONING:\s*(.+)", response)
            reason_text = reasoning.group(1)[:80] if reasoning else "(no reasoning)"
            print(f"  {crew_name} → {vote} {correct_mark} | {reason_text}")

        requests.post(
            f"{env_url}/multi/vote",
            json={"game_id": game_id, "player_name": crew_name, "vote_target": vote},
            timeout=10,
        )

    result = requests.get(f"{env_url}/multi/resolve/{game_id}", timeout=10).json()

    if verbose:
        print(f"\n  >> Ejected: {result.get('ejected')} | Correct: {result.get('correct')}")
        print(f"  >> {result.get('message')}")

    return {
        "correct": result.get("correct", False),
        "ejected": result.get("ejected"),
        "impostor_names": impostor_names,
        "votes": votes,
        "majority_reward": result.get("majority_reward", -0.5),
        "player_rewards": result.get("player_rewards", {}),
    }


# Evaluate 20 games
print("=== MULTI-AGENT EVALUATION (with debate) ===")
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

correct = 0
total = 20
for g in range(total):
    r = run_multiagent_episode(model, tokenizer)
    if r is None:
        total -= 1
        continue
    if r["correct"]:
        correct += 1

print(f"\nAccuracy: {correct}/{total} = {correct/max(1,total):.1%}")
print(f"(Random baseline: ~16.7% with 6 players)")

In [ ]:
print("=== LIVE DEMO: Full Among Us Game with Debate ===")
print()
result = run_multiagent_episode(model, tokenizer, verbose=True)

In [ ]:
if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    print(f"Saving model to {OUTPUT_HF}...")
    model.save_pretrained_merged(
        "multiagent_model_merged", tokenizer, save_method="merged_16bit"
    )
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(OUTPUT_HF, repo_type="model", exist_ok=True)
    api.upload_folder(
        folder_path="multiagent_model_merged",
        repo_id=OUTPUT_HF,
        repo_type="model",
    )
    print(f"Model saved to https://huggingface.co/{OUTPUT_HF}")
else:
    model.save_pretrained("./multiagent_model")
    tokenizer.save_pretrained("./multiagent_model")
    print("Model saved locally to ./multiagent_model")